# Make the Lake Slippery: Solution

A solution of the practice task *Make the lake slippery*, in this folder's [README](../README.md#practice): try the task before reading it. The code mirrors the notebook [Q-learning on FrozenLake](../frozen_lake.ipynb), with a seed for every run so that the numbers can be reproduced.

## The Slippery Lake

In [ ]:
import gymnasium as gym

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
transitions = env.unwrapped.P
transitions[0][2]

[(0.33333333333333337, 4, 0, False),
 (0.3333333333333333, 1, 0, False),
 (0.33333333333333337, 0, 0, False)]

`env.unwrapped.P[state][action]` lists the outcomes of an action as `(probability, next state, reward, terminated)`. RIGHT from the start goes right, to state 1, one time in three; otherwise it slips perpendicular to the intended direction: DOWN to state 4, or UP into the wall, which leaves the agent at the start. The whole table is known here, which lets us compute the best possible policy exactly: something an agent that learns from experience doesn't get to do.

## Question 1: How Often, and How Much It Varies

The notebook's training, as one function, with a seed for each run:

In [ ]:
import random

import numpy as np


def train(learning_rate, gamma, n_training_episodes, seed, max_steps=99, max_epsilon=1.0, min_epsilon=0.05,
          decay_rate=0.0005):
    random.seed(seed)
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    qtable = np.zeros((env.observation_space.n, env.action_space.n))
    for episode in range(n_training_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        for step in range(max_steps):
            if random.uniform(0, 1) > epsilon:
                action = np.argmax(qtable[state])
            else:
                action = env.action_space.sample()
            new_state, reward, terminated, truncated, info = env.step(action)
            qtable[state][action] += learning_rate * (
                reward + gamma * np.max(qtable[new_state]) - qtable[state][action]
            )
            if terminated or truncated:
                break
            state = new_state
    return qtable

Its evaluation, with the greedy policy, on an environment of its own, so that every Q-table is evaluated on the same sequence of slips:

In [ ]:
def success_rate(qtable, n_eval_episodes=1000, seed=1000, max_steps=99):
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
    env.reset(seed=seed)
    successes = 0
    for episode in range(n_eval_episodes):
        state, info = env.reset()
        for step in range(max_steps):
            state, reward, terminated, truncated, info = env.step(np.argmax(qtable[state]))
            if terminated or truncated:
                successes += reward
                break
    return successes / n_eval_episodes

Five runs with the notebook's settings: α 0.7, γ 0.95, 10,000 episodes.

In [ ]:
seeds = range(5)
notebook_tables = [train(0.7, 0.95, 10_000, seed) for seed in seeds]
[success_rate(qtable) for qtable in notebook_tables]

[0.729, 0.309, 0.718, 0.379, 0.619]

Between 0.31 and 0.73, with the same code and the same settings: on a slippery lake, one run says little.

The evaluation adds noise of its own. The first of those Q-tables, evaluated five times over 100 episodes, as the notebook does:

In [ ]:
[success_rate(notebook_tables[0], n_eval_episodes=100, seed=evaluation_seed) for evaluation_seed in range(5)]

[0.63, 0.75, 0.71, 0.82, 0.79]

The same Q-table scores between 0.63 and 0.82, around the 0.729 that 1,000 episodes measure. Over $n$ episodes, a success rate near 0.7 is estimated within about $\pm 1.96 \sqrt{0.7 \cdot 0.3 / n}$ (95%): ±9 points with 100 episodes, ±3 with 1,000, which is why the task asks for 1,000. The spread between the runs above, from 0.31 to 0.73, is far wider than ±3 points: it comes from training, not from the evaluation.

## The Ceiling

Since the transition table is known, **value iteration** computes the optimal action values $Q^*$: it applies the Bellman equation of [notebook 2](../bellman_equation.ipynb) as an assignment, to every state and action, until the values stop changing. On a slippery lake, its right-hand side is an average over the three outcomes of each action.

In [ ]:
def action_values(state_values, gamma):
    q = np.zeros((16, 4))
    for state in range(16):
        for action in range(4):
            for probability, next_state, reward, terminated in transitions[state][action]:
                q[state, action] += probability * (reward + gamma * state_values[next_state] * (not terminated))
    return q


def optimal_action_values(gamma, iterations=2000):
    state_values = np.zeros(16)
    for _ in range(iterations):
        state_values = action_values(state_values, gamma).max(axis=1)
    return action_values(state_values, gamma)


optimal_values = {gamma: optimal_action_values(gamma) for gamma in [0.95, 0.99]}

The success rate of a policy within the 99-step limit, computed exactly: the probability of reaching the goal in at most $n$ steps, built up one step at a time from $n = 0$.

In [ ]:
def success_probability(policy, max_steps=99):
    success_within = np.zeros(16)
    for _ in range(max_steps):
        success_within_one_more = np.zeros(16)
        for state in range(16):
            for probability, next_state, reward, terminated in transitions[state][policy[state]]:
                success_within_one_more[state] += probability * (
                    reward + success_within[next_state] * (not terminated)
                )
        success_within = success_within_one_more
    return success_within[0]


optimal_policies = {gamma: q.argmax(axis=1) for gamma, q in optimal_values.items()}
{gamma: round(success_probability(policy), 3) for gamma, policy in optimal_policies.items()}

{0.95: np.float64(0.728), 0.99: np.float64(0.738)}

No agent can beat 0.738 within 99 steps: that is the task's 74%. The optimal policies of the two values of γ differ only at state 2, on the top row: γ 0.95 chooses LEFT, which slips DOWN to state 6 one time in three, a quicker way down; γ 0.99 chooses UP, which stays on the top row. Discounting makes a slow, safe route worth less, so γ 0.95 accepts a little more risk (0.728).

Without the step limit, the γ 0.99 policy would do better:

In [ ]:
round(success_probability(optimal_policies[0.99], max_steps=10_000), 3)

np.float64(0.824)

82.4%: almost 9 points are lost to the 99-step limit alone, because the safe policy spends many moves bumping into walls, and some episodes run out of steps. The other 18% falls into holes, and no policy can avoid that. The actions that can't fall into a hole, state by state, show why:

In [ ]:
action_names = ["LEFT", "DOWN", "RIGHT", "UP"]
holes = {5, 7, 11, 12}
goal = 15

{
    state: [
        action_names[action]
        for action in range(4)
        if all(next_state not in holes for _, next_state, _, _ in transitions[state][action])
    ]
    for state in range(16)
    if state not in holes and state != goal
}

{0: ['LEFT', 'DOWN', 'RIGHT', 'UP'],
 1: ['UP'],
 2: ['LEFT', 'DOWN', 'RIGHT', 'UP'],
 3: ['UP'],
 4: ['LEFT'],
 6: [],
 8: ['UP'],
 9: ['DOWN'],
 10: ['LEFT'],
 13: ['RIGHT'],
 14: ['LEFT', 'DOWN', 'RIGHT', 'UP']}

State 6 has no safe action: every move slips into hole 5 or hole 7 one time in three. The safe route goes down the left column (0, 4, 8, 9, 13, 14), but at state 9 the only safe action, DOWN, slips RIGHT to state 10 one time in three, and from state 10 the only safe action, LEFT, slips UP to state 6 one time in three. Every route to the goal passes state 9 or 10, so no policy can close that leak. The top row is no way round it: from it, the only ways down are state 4, back to the left column, state 6, or a hole.

## Question 2: Which Settings Reach the Ceiling

Five runs of each setting: one change at a time, then the two that matter together. This cell takes about two minutes.

In [ ]:
import pandas as pd

settings = {
    "notebook (α 0.7, γ 0.95, 10,000 episodes)": (0.7, 0.95, 10_000),
    "α 0.1": (0.1, 0.95, 10_000),
    "γ 0.99": (0.7, 0.99, 10_000),
    "50,000 episodes": (0.7, 0.95, 50_000),
    "α 0.1 and γ 0.99": (0.1, 0.99, 10_000),
}
trained_tables = {
    name: [train(learning_rate, gamma, n_episodes, seed) for seed in seeds]
    for name, (learning_rate, gamma, n_episodes) in settings.items()
}
results = pd.DataFrame(
    {name: [success_rate(qtable) for qtable in tables] for name, tables in trained_tables.items()},
    index=[f"seed {seed}" for seed in seeds],
).T
results["mean"] = results.mean(axis=1)
results.round(3)

,seed 0,seed 1,seed 2,seed 3,seed 4,mean
"notebook (α 0.7, γ 0.95, 10,000 episodes)",0.729,0.309,0.718,0.379,0.619,0.551
α 0.1,0.721,0.297,0.729,0.722,0.721,0.638
γ 0.99,0.225,0.401,0.584,0.165,0.584,0.392
"50,000 episodes",0.703,0.220,0.729,0.584,0.601,0.567
α 0.1 and γ 0.99,0.728,0.718,0.718,0.728,0.728,0.724


- **No single change is enough.** α 0.1 alone reaches the γ 0.95 ceiling (0.728) in four runs out of five, but seed 1 fails at 0.297 (see *One Failure of α 0.1*, below). γ 0.99 alone and five times more episodes don't help: with α 0.7 the runs spread from 0.17 to 0.73, and with five runs the differences between those means are within that spread, so they aren't evidence that either change makes things worse.
- **α 0.1 and γ 0.99 together reach 0.72–0.73 in every run:** the 0.738 ceiling, within the ±3 points of a 1,000-episode evaluation.
- **Several runs per setting are part of the answer.** Judged by seed 0 alone, the notebook's settings (0.729) would look as good as the best ones.

### Why the Learning Rate Matters

On the slippery lake the target of an update, $r + \gamma \max_{a'} Q(s', a')$, is random: it depends on where the agent slipped. With a constant learning rate, $Q(s, a)$ is an exponentially weighted average of the targets it has seen, with weight $\alpha (1 - \alpha)^k$ on the target of $k$ updates ago. If the targets were independent, its variance would be $\frac{\alpha}{2 - \alpha}$ times the variance of one target: 0.54 with α 0.7, 0.053 with α 0.1. With α 0.7 the estimate is essentially the last two or three targets, however long the training goes on, which is why 50,000 episodes don't help. On the deterministic lake every target of an action is the same, so α didn't matter there.

What that noise costs depends on the **margin** between the best and the second-best action of each state: when the error is larger than the margin, the greedy choice between the two is a coin toss. The margins of the optimal values in the states where the agent decides (state 6 is left out: LEFT and RIGHT tie there, mirror images of each other):

In [ ]:
decision_states = [state for state in range(16) if state not in holes and state not in (goal, 6)]


def margins(q):
    sorted_values = np.sort(q[decision_states], axis=1)
    return sorted_values[:, -1] - sorted_values[:, -2]


pd.DataFrame({f"γ {gamma}": margins(q) for gamma, q in optimal_values.items()}, index=decision_states).round(3)

,γ 0.95,γ 0.99
0,0.008,0.014
1,0.049,0.155
2,0.007,0.033
3,0.042,0.151
4,0.057,0.179
8,0.066,0.184
9,0.086,0.195
10,0.056,0.118
13,0.119,0.212
14,0.033,0.042


γ 0.99 has larger margins in every state (median 0.153 against 0.052): an agent that values the far future more loses more by falling into a hole, so safe and risky actions differ more. The smallest margins, at states 0 and 2, are where a noisy agent is most likely to choose wrong.

Next, the error of the learned values relative to those margins (for each decision state, the largest error over its four actions divided by its margin; the median over the states and then over the five runs), and the share of decision states in which the learned greedy action is the optimal one (the mean over the runs):

In [ ]:
def relative_error(qtable, gamma):
    errors = np.abs(qtable - optimal_values[gamma])[decision_states].max(axis=1)
    return np.median(errors / margins(optimal_values[gamma]))


def optimal_choices(qtable, gamma):
    learned_actions = qtable[decision_states].argmax(axis=1)
    return np.mean(learned_actions == optimal_values[gamma][decision_states].argmax(axis=1))


pd.DataFrame(
    {
        "error / margin": [
            np.median([relative_error(qtable, settings[name][1]) for qtable in tables])
            for name, tables in trained_tables.items()
        ],
        "optimal choices": [
            np.mean([optimal_choices(qtable, settings[name][1]) for qtable in tables])
            for name, tables in trained_tables.items()
        ],
    },
    index=list(trained_tables),
).round(2)

,error / margin,optimal choices
"notebook (α 0.7, γ 0.95, 10,000 episodes)",2.96,0.84
α 0.1,0.90,0.88
γ 0.99,2.06,0.84
"50,000 episodes",2.81,0.80
α 0.1 and γ 0.99,0.62,1.00


With α 0.7 the errors are two to three times the margins, whatever γ or the number of episodes, and roughly one decision in six is wrong. α 0.1 brings the errors down to about the margins of γ 0.95; with γ 0.99, whose margins are larger, they fall below them (0.62), and the agent chooses the optimal action in every decision state of every run. So γ 0.99 helps in two ways: its optimal policy is a little safer (0.738 against 0.728), and its larger margins are easier to learn. Neither shows while the noise of α 0.7 dominates.

### One Failure of α 0.1

The failed run of α 0.1 (seed 1), against the optimal values, in the states where its greedy choice differs:

In [ ]:
failed_run = trained_tables["α 0.1"][1]
wrong_states = [
    state for state in decision_states if failed_run[state].argmax() != optimal_values[0.95][state].argmax()
]
pd.concat(
    {
        "learned": pd.DataFrame(failed_run[wrong_states], index=wrong_states, columns=action_names),
        "optimal": pd.DataFrame(optimal_values[0.95][wrong_states], index=wrong_states, columns=action_names),
    },
    axis=1,
).round(3)

learned                      optimal                     
      LEFT   DOWN  RIGHT     UP    LEFT   DOWN  RIGHT     UP
2    0.128  0.132  0.125  0.117   0.153  0.147  0.146  0.140
9    0.291  0.299  0.328  0.214   0.247  0.375  0.289  0.213
14   0.502  0.580  0.669  0.579   0.518  0.724  0.690  0.622

Its costly mistake is at state 9, on the safe route: RIGHT slips UP into hole 5 one time in three, while DOWN can't fall. Its estimates of the best actions are too low (DOWN at state 9: 0.299 against 0.375; DOWN at state 14: 0.580 against 0.724), which suggests training that hasn't finished. If so, more episodes should fix it:

In [ ]:
success_rate(train(0.1, 0.95, 50_000, seed=1))

0.514

They don't: 0.514. The table of errors explains it better. With α 0.1 and γ 0.95 the errors are about as large as the margins (0.90), so roughly one decision in eight is wrong, and in this run one of them falls on state 9. A constant α keeps the same level of noise however long it trains, so more episodes give a different run, not a less noisy one. With γ 0.99 the margin at state 9 more than doubles (0.195 against 0.086), and α 0.1 becomes enough. The textbook alternative, not tried here, is a learning rate that decreases with the number of visits to each state and action: Q-learning then converges to $Q^*$ (Watkins and Dayan, 1992), at the price of slower learning.

## Question 3: The Start Action

In [ ]:
pd.Series(optimal_values[0.99][0], index=action_names).round(3)

LEFT     0.542
DOWN     0.528
RIGHT    0.528
UP       0.522
dtype: float64

Every start action mixes the same three outcomes: staying in the corner (a bump into a wall), state 4 down the left column, or state 1 on the top row. LEFT gives stay, stay or state 4; DOWN and RIGHT give stay, state 4 or state 1, the same mix and so the same value; UP gives stay, stay or state 1. The values only rank how much each action prefers state 4 to state 1. State 4 is worth more because the left column is the safe route, while from the top row the agent has to come back or go down through state 6. LEFT, which points away from the goal, is best because the walls turn two of its three outcomes into a harmless wait, and the third is the good one.

In [ ]:
{name: [action_names[qtable[0].argmax()] for qtable in tables] for name, tables in trained_tables.items()}

{'notebook (α 0.7, γ 0.95, 10,000 episodes)': ['LEFT',
  'LEFT',
  'LEFT',
  'LEFT',
  'LEFT'],
 'α 0.1': ['LEFT', 'LEFT', 'LEFT', 'LEFT', 'LEFT'],
 'γ 0.99': ['LEFT', 'RIGHT', 'LEFT', 'LEFT', 'LEFT'],
 '50,000 episodes': ['LEFT', 'RIGHT', 'LEFT', 'LEFT', 'LEFT'],
 'α 0.1 and γ 0.99': ['LEFT', 'LEFT', 'LEFT', 'LEFT', 'LEFT']}

Almost every run finds LEFT, despite the smallest margin of all (0.014); the two that choose RIGHT are α 0.7 runs. RIGHT costs little here: it is worth 0.528 against 0.542.

## What a Good Answer Says

- **Question 1.** The success rate varies widely between runs (0.31 to 0.73 with the notebook's settings), so one run proves nothing; 100 evaluation episodes add ±9 points of their own, 1,000 about ±3.
- **Question 2.** No policy reaches the goal every time: the safe route leaks through states 9, 10 and 6, and the step limit costs about 9 points more, so about 74% is the target. The learning rate is what matters: slips make every update a random sample, which a high α follows instead of averaging, and more episodes can't fix that. With a low α, γ 0.99 then reaches the ceiling in every run. A good answer shows several runs per setting, and changes one thing at a time before combining them.
- **Question 3.** LEFT, away from the goal: against the walls it can only wait or slip DOWN, so it never sends the agent to the top row. A good answer explains it with the slip rule, not as a mistake of the agent.
- **Watch for** a Q-table that wasn't re-created between runs (training then continues from the last one), and conclusions drawn from one run or from 100 evaluation episodes.